# Pitcher WAR Pipeline - Complete Workflow

**Purpose:** Train pitcher WAR models from scratch and generate 2025 projections

**Last Updated:** 2025-10-06

---

## Pipeline Overview
1. Load historical data (2016-2024) for training
2. Run sklearn pipeline (filters → transformers → features)
3. Split by role (starter/reliever/swing)
4. Train role-based ensemble models
5. Generate 2025 predictions with ROS projections
6. Validate performance (MAE, R², residuals)
7. Feature importance analysis
8. Save models and predictions

In [1]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
project_root = Path('.').absolute().parent.parent.parent
sys.path.insert(0, str(project_root))

from new_pipeline.notebooks.shared.pipeline_runner import (
    load_historical_data,
    load_current_season_data,
    run_data_pipeline,
    generate_predictions,
    calculate_metrics,
    split_by_role
)
from new_pipeline.notebooks.shared.plotting_utils import (
    create_actual_vs_predicted,
    create_residual_plot,
    create_feature_importance
)
from new_pipeline.notebooks.shared.analysis_utils import (
    calculate_elite_performance,
    analyze_errors_by_group
)
from new_pipeline.models import PitcherRoleEnsemble
from new_pipeline.common.constants import PITCHER_MODEL_FEATURES

print("Imports successful!")
print(f"Pitcher features: {len(PITCHER_MODEL_FEATURES)}")

23:54:16 - new_pipeline.common.logging_config - INFO - Logging module initialized for new_pipeline
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!
Pitcher features: 13


In [2]:
# Cell 2: Load Historical Training Data

print("Loading historical pitcher data (2016-2024)...")

pitcher_historical = load_historical_data(
    player_type='pitcher',
    years=range(2016, 2025)
)

print(f"\nLoaded {len(pitcher_historical)} pitcher-seasons")
print(f"Years: {sorted(pitcher_historical['Year'].unique())}")
print(f"\nSample columns: {list(pitcher_historical.columns[:10])}")

Loading historical pitcher data (2016-2024)...

Loaded 7237 pitcher-seasons
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Sample columns: ['Name', 'Team', 'W', 'L', 'SV', 'G', 'GS', 'IP', 'K/9', 'BB/9']


In [3]:
# Cell 3: Run Data Pipeline

print("Running sklearn pipeline...")
print("Steps: Filters → Feature Loading → Composites → Imputation → Validation → Selection → Normalization")

pitcher_processed = run_data_pipeline(
    pitcher_historical,
    player_type='pitcher'
)

print(f"\nPipeline complete!")
print(f"Processed {len(pitcher_processed)} qualified pitchers")
print(f"Features: {len(PITCHER_MODEL_FEATURES)}")
print(f"\nFeature list: {PITCHER_MODEL_FEATURES}")
print(f"\nTarget: WAR_per_162 (range: {pitcher_processed['WAR_per_162'].min():.2f} to {pitcher_processed['WAR_per_162'].max():.2f})")

23:54:21 - new_pipeline.common.transformers.filters - WARNING - TwoWayPlayerFilter: Missing required columns (IP, GS, PA), marking all as False
23:54:22 - new_pipeline.common.transformers.filters - INFO - IPFilter: Removed 1585 pitchers (position players / insufficient sample, full season)
23:54:22 - new_pipeline.common.transformers.pitcher_features - INFO - Loading pitcher features...


Running sklearn pipeline...
Steps: Filters → Feature Loading → Composites → Imputation → Validation → Selection → Normalization


23:54:24 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 11 pitcher feature sets (35 total columns)
23:54:24 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
23:54:24 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 6 composite features
23:54:24 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 37 features
23:54:24 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 2881 missing values
23:54:24 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 50.00] outside expected [0, 25]
  - Feature 'ERA' range [0.00, 53.13] outside expected [0, 15]
  - Feature 'GB%' range [0.00, 103.03] outside expected [20, 80]
23:54:24 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 13 features + 9 met


Pipeline complete!
Processed 5652 qualified pitchers
Features: 13

Feature list: ['BB%', 'K%', 'ERA', 'GB%', 'SwStr%', 'WPA/LI', 'damage_control_ratio', 'Opportunity_Success', 'strikeout_efficiency', 'contact_management', 'strikeout_contact_quality', 'Statcast_Launch_Quality_Index', 'Running_Control']

Target: WAR_per_162 (range: -34.59 to 10.39)


In [4]:
# Cell 4: Split by Role

print("Splitting pitchers by role...")

role_splits = split_by_role(pitcher_processed)

print(f"\nStarters: {len(role_splits['Starter'])} ({len(role_splits['Starter'])/len(pitcher_processed)*100:.1f}%)")
print(f"Relievers: {len(role_splits['Reliever'])} ({len(role_splits['Reliever'])/len(pitcher_processed)*100:.1f}%)")
print(f"Swing: {len(role_splits['Swing'])} ({len(role_splits['Swing'])/len(pitcher_processed)*100:.1f}%)")

print("\nRole classification criteria:")
print("  Starter: GS/G > 0.7")
print("  Reliever: GS/G < 0.1")
print("  Swing: 0.1 <= GS/G <= 0.7")

Splitting pitchers by role...

Starters: 1892 (33.5%)
Relievers: 2895 (51.2%)
Swing: 865 (15.3%)

Role classification criteria:
  Starter: GS/G > 0.7
  Reliever: GS/G < 0.1
  Swing: 0.1 <= GS/G <= 0.7


In [5]:
# Cell 5: Prepare Training Data

print("Preparing training data...")

# Extract features and target
X_train = pitcher_processed[PITCHER_MODEL_FEATURES].values
y_train = pitcher_processed['WAR_per_162'].values

# Create role labels
pitcher_processed['GS_per_G'] = pitcher_processed['GS'] / pitcher_processed['G'].replace(0, 1)

def get_role(row):
    if row['GS_per_G'] > 0.7:
        return 'starter'
    elif row['GS_per_G'] < 0.1:
        return 'reliever'
    else:
        return 'swing'

roles = pitcher_processed.apply(get_role, axis=1).values

print(f"Training data shape: {X_train.shape}")
print(f"Target shape: {y_train.shape}")
print(f"Role distribution: {pd.Series(roles).value_counts().to_dict()}")

Preparing training data...
Training data shape: (5652, 13)
Target shape: (5652,)
Role distribution: {'reliever': 2895, 'starter': 1892, 'swing': 865}


In [6]:
# Cell 6: Train Role-Based Ensemble Models

print("Training pitcher role-based ensembles...")
print("  Each role gets: RandomForest + Keras + MultiQuantileHistGB")
print("\nThis may take 3-5 minutes...\n")

pitcher_model = PitcherRoleEnsemble()
pitcher_model.fit(X_train, y_train, roles)

print("\nTraining complete!")
print("Models trained:")
print("  - Starter ensemble (3 models)")
print("  - Reliever ensemble (3 models)")
print("  - Swing ensemble (3 models)")

23:54:24 - new_pipeline.models.pitcher_ensemble - INFO - Training starter ensemble (1892 samples)...
23:54:24 - new_pipeline.models.pitcher_ensemble - INFO -   Training ExtraTrees...
23:54:24 - new_pipeline.models.pitcher_ensemble - INFO -   Training Keras (AdamW + Swish + BatchNorm)...
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training pitcher role-based ensembles...
  Each role gets: RandomForest + Keras + MultiQuantileHistGB

This may take 3-5 minutes...


Epoch 17: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 27: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 32: early stopping
Restoring model weights from the end of the best epoch: 7.


23:54:36 - new_pipeline.models.pitcher_ensemble - INFO -   Training MultiQuantileHistGB...
23:54:43 - new_pipeline.models.pitcher_ensemble - INFO -   Starter ensemble training complete
23:54:43 - new_pipeline.models.pitcher_ensemble - INFO - Training reliever ensemble (2895 samples)...
23:54:43 - new_pipeline.models.pitcher_ensemble - INFO -   Training ExtraTrees...
23:54:43 - new_pipeline.models.pitcher_ensemble - INFO -   Training Keras (AdamW + Swish + BatchNorm)...
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Epoch 13: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 23: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 28: early stopping
Restoring model weights from the end of the best epoch: 3.


23:55:02 - new_pipeline.models.pitcher_ensemble - INFO -   Training MultiQuantileHistGB...
23:55:07 - new_pipeline.models.pitcher_ensemble - INFO -   Reliever ensemble training complete
23:55:07 - new_pipeline.models.pitcher_ensemble - INFO - Training swing ensemble (865 samples)...
23:55:07 - new_pipeline.models.pitcher_ensemble - INFO -   Training ExtraTrees...
23:55:07 - new_pipeline.models.pitcher_ensemble - INFO -   Training Keras (AdamW + Swish + BatchNorm)...
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Epoch 31: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 41: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 46: early stopping
Restoring model weights from the end of the best epoch: 21.


23:55:24 - new_pipeline.models.pitcher_ensemble - INFO -   Training MultiQuantileHistGB...
23:55:28 - new_pipeline.models.pitcher_ensemble - INFO -   Swing ensemble training complete



Training complete!
Models trained:
  - Starter ensemble (3 models)
  - Reliever ensemble (3 models)
  - Swing ensemble (3 models)


In [7]:
# Cell 7: Training Set Validation

print("Validating on training data...")

y_pred_train = pitcher_model.predict(X_train, roles)

metrics = calculate_metrics(y_train, y_pred_train)

print("\n" + "="*50)
print("TRAINING METRICS")
print("="*50)
print(f"MAE:  {metrics['MAE']:.3f}")
print(f"RMSE: {metrics['RMSE']:.3f}")
print(f"R²:   {metrics['R²']:.3f}")

# Elite pitcher performance
elite_metrics = calculate_elite_performance(y_train, y_pred_train, threshold=5.0)
print(f"\nElite (>5 WAR) MAE: {elite_metrics['elite_MAE']:.3f} ({elite_metrics['elite_count']} pitchers)")

print("="*50)

Validating on training data...

TRAINING METRICS
MAE:  1.559
RMSE: 2.280
R²:   0.259

Elite (>5 WAR) MAE: 3.594 (197 pitchers)


In [8]:
# Cell 8: Load 2025 Data for Predictions

print("Loading 2025 current season data...")

pitcher_2025_raw = load_current_season_data('pitcher', year=2025)

print(f"Loaded {len(pitcher_2025_raw)} pitchers (raw)")

# Run pipeline
print("\nProcessing through pipeline...")
pitcher_2025_processed = run_data_pipeline(pitcher_2025_raw, player_type='pitcher')

print(f"Processed {len(pitcher_2025_processed)} qualified pitchers")

23:55:30 - new_pipeline.common.transformers.filters - WARNING - TwoWayPlayerFilter: Missing required columns (IP, GS, PA), marking all as False
23:55:30 - new_pipeline.common.transformers.filters - INFO - IPFilter: Removed 157 pitchers (position players / insufficient sample, partial season)
23:55:30 - new_pipeline.common.transformers.pitcher_features - INFO - Loading pitcher features...


Loading 2025 current season data...
Loading partial season data: fangraphs_pitchers_2025_firsthalf.csv
Loaded 754 pitchers (raw)

Processing through pipeline...


23:55:30 - new_pipeline.common.transformers.pitcher_features - INFO - Loaded 11 pitcher feature sets (35 total columns)
23:55:30 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Calculating pitcher composite features...
23:55:30 - new_pipeline.common.transformers.pitcher_composite_transformer - INFO - Added 6 composite features
23:55:30 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Learned replacement values for 37 features
23:55:30 - new_pipeline.common.transformers.validators - INFO - MissingValueImputer: Imputed 439 missing values


23:55:30 - new_pipeline.common.transformers.validators - WARNING - FeatureValidator found issues:
  - Feature 'BB%' range [0.00, 26.92] outside expected [0, 25]
  - Feature 'ERA' range [0.00, 19.86] outside expected [0, 15]
  - Feature 'GB%' range [11.11, 74.71] outside expected [20, 80]
23:55:30 - new_pipeline.common.transformers.feature_selector - INFO - FeatureSelector: Selected 13 features + 9 metadata columns
23:55:30 - new_pipeline.common.transformers.normalizers - INFO - WARNormalizer: Added 'WAR_per_162' column


Processed 597 qualified pitchers


In [9]:
# Cell 9: Generate 2025 Predictions

print("Generating 2025 predictions...")

pitcher_predictions = generate_predictions(
    pitcher_2025_processed,
    pitcher_model,
    player_type='pitcher'
)

print(f"\nGenerated predictions for {len(pitcher_predictions)} pitchers")
print("\nPrediction columns added:")
pred_cols = [c for c in pitcher_predictions.columns if 'Predicted' in c or 'ROS' in c or 'Total' in c]
print(f"  {pred_cols}")

print("\nTop 10 Projected WAR (Full Season):")
top_10 = pitcher_predictions.nlargest(10, 'Total_Projected_WAR')[['Name', 'Team', 'IP', 'Current_WAR', 'ROS_WAR', 'Total_Projected_WAR']]
print(top_10.to_string(index=False))

c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Generating 2025 predictions...


c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\nairs\Documents\GithubProjects\oWAR\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(



Generated predictions for 597 pitchers

Prediction columns added:
  ['Predicted_WAR_per_162', 'ROS_WAR', 'Total_Projected_WAR']

Top 10 Projected WAR (Full Season):
              Name Team    IP  Current_WAR  ROS_WAR  Total_Projected_WAR
      Tarik Skubal  DET 121.0     4.772298 2.928212             7.700510
   Garrett Crochet  BOS 129.1     4.329465 2.045205             6.374670
       Paul Skenes  PIT 121.0     4.012076 1.800063             5.812139
      Zack Wheeler  PHI 122.0     3.695160 1.973982             5.669141
Cristopher Sánchez  PHI 115.0     3.254897 1.898711             5.153608
    Nathan Eovaldi  TEX  91.0     2.997249 2.104537             5.101786
        Logan Webb  SFG 125.2     3.456942 1.644627             5.101568
        Kris Bubic  KCR 108.2     3.204435 1.753041             4.957477
        Chris Sale  ATL  89.1     2.617011 2.220746             4.837757
      Hunter Brown  HOU 115.0     2.913486 1.897722             4.811208


In [10]:
# Cell 10: Actual vs Predicted Plot

# Add role for coloring
pitcher_processed['Role'] = pd.Series(roles, index=pitcher_processed.index)

fig_scatter = create_actual_vs_predicted(
    y_true=y_train,
    y_pred=y_pred_train,
    color_by=pitcher_processed['Role'].values
)

fig_scatter.update_layout(title="Pitcher WAR: Actual vs Predicted (Training Set)")
fig_scatter.show()

In [11]:
# Cell 11: Residual Analysis

residuals = y_train - y_pred_train

fig_residuals = create_residual_plot(
    residuals=residuals,
    color_by=pitcher_processed['Role'].values
)

fig_residuals.update_layout(title="Pitcher Residual Distribution by Role")
fig_residuals.show()

print(f"\nResidual statistics:")
print(f"  Mean: {residuals.mean():.3f}")
print(f"  Std: {residuals.std():.3f}")
print(f"  Min: {residuals.min():.3f}")
print(f"  Max: {residuals.max():.3f}")


Residual statistics:
  Mean: -0.121
  Std: 2.277
  Min: -32.320
  Max: 8.871


In [12]:
# Cell 12: Feature Importance

print("Extracting feature importance from RandomForest components...")

# Get importance from starter model (largest sample)
if hasattr(pitcher_model.models['starter'], 'rf_model'):
    importance_values = pitcher_model.models['starter'].rf_model.feature_importances_
    importance_dict = dict(zip(PITCHER_MODEL_FEATURES, importance_values))
    
    fig_importance = create_feature_importance(importance_dict, top_n=13)
    fig_importance.update_layout(title="Pitcher Feature Importance (Starter Model - RandomForest)")
    fig_importance.show()
    
    # Print top features
    print("\nTop 5 Most Important Features:")
    sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)
    for feat, imp in sorted_importance[:5]:
        print(f"  {feat}: {imp:.4f}")
else:
    print("Feature importance not available (model doesn't have rf_model attribute)")

Extracting feature importance from RandomForest components...
Feature importance not available (model doesn't have rf_model attribute)


In [13]:
# Cell 13: Error Analysis by Role

print("Analyzing errors by role...")

role_errors = analyze_errors_by_group(
    residuals=residuals,
    groups=pitcher_processed['Role'].values
)

print("\n" + "="*50)
print("ERROR ANALYSIS BY ROLE")
print("="*50)
for role, metrics in role_errors.items():
    print(f"\n{role.upper()}:")
    print(f"  Count: {metrics['count']}")
    print(f"  MAE: {metrics['MAE']:.3f}")
    print(f"  RMSE: {metrics['RMSE']:.3f}")
    print(f"  Mean Error: {metrics['mean_error']:.3f}")
    print(f"  Std Error: {metrics['std_error']:.3f}")
print("="*50)

Analyzing errors by role...

ERROR ANALYSIS BY ROLE

RELIEVER:
  Count: 2895
  MAE: 1.593
  RMSE: 2.132
  Mean Error: -0.039
  Std Error: 2.132

STARTER:
  Count: 1892
  MAE: 1.516
  RMSE: 2.513
  Mean Error: -0.246
  Std Error: 2.501

SWING:
  Count: 865
  MAE: 1.539
  RMSE: 2.223
  Mean Error: -0.122
  Std Error: 2.220


In [14]:
# Cell 14: Save Models and Predictions

import joblib

# Save model
model_path = project_root / 'models' / 'pitcher_role_ensemble_2025.pkl'
model_path.parent.mkdir(exist_ok=True)
joblib.dump(pitcher_model, model_path)
print(f"Model saved to: {model_path}")

# Save predictions
predictions_path = project_root / 'predictions' / 'pitcher_predictions_2025.csv'
predictions_path.parent.mkdir(exist_ok=True)
pitcher_predictions.to_csv(predictions_path, index=False)
print(f"Predictions saved to: {predictions_path}")

print("\n" + "="*50)
print("PITCHER PIPELINE COMPLETE!")
print("="*50)
print(f"Trained on {len(pitcher_processed)} historical pitcher-seasons")
print(f"Generated predictions for {len(pitcher_predictions)} 2025 pitchers")
print(f"Overall MAE: {metrics['MAE']:.3f}")
print(f"Overall R²: {metrics['R²']:.3f}")
print("="*50)

Model saved to: c:\Users\nairs\Documents\GithubProjects\oWAR\models\pitcher_role_ensemble_2025.pkl
Predictions saved to: c:\Users\nairs\Documents\GithubProjects\oWAR\predictions\pitcher_predictions_2025.csv

PITCHER PIPELINE COMPLETE!
Trained on 5652 historical pitcher-seasons
Generated predictions for 597 2025 pitchers
Overall MAE: 1.539


KeyError: 'R²'